In [70]:
from flask import Flask, render_template_string
import copy
import heapq

app = Flask(__name__)

NOH = 2
NOL = 2
# Initialize robots' positions and blocks they are holding

# BOX_Current = [['a', 'c', 'e'], ['b', 'f', 'd', 'g']]
# BOX_Desired = [['d', 'g', 'b'], ['f', 'e', 'a', 'c']]

BOX_Current = [['a', 'b', 'c'], ['d', 'e', 'f', 'g']]
BOX_Desired = [['c', 'f', 'a', 'd'], ['e', 'b', 'g']]


# print("Enter number of boxes in .... [Current State]")
# for i in range(NOL):
#   BOX_Current.append(input(f'L{i} : ').split())

# print("Enter number of boxes in .... [Final State]")
# for i in range(NOL):
#   BOX_Desired.append(input(f'L{i} : ').split())

BOX_Current.append({'R1' : {'block' :None, 'location' : 0}, 'R2' : {'block' : None, 'location' : 1}})

visited_states = []


def expand_children(state):
  results = []
  # R1 Pickup
  Copy_BOX_Current = copy.deepcopy(state)
  if Copy_BOX_Current[2]['R1']['block'] is None and len(Copy_BOX_Current[Copy_BOX_Current[2]['R1']['location']]) > 0:
    Copy_BOX_Current[2]['R1']['block'] = Copy_BOX_Current[Copy_BOX_Current[2]['R1']['location']].pop()
    results.append(Copy_BOX_Current)

  # R1 DropOff
  Copy_BOX_Current = copy.deepcopy(state)
  if Copy_BOX_Current[2]['R1']['block'] is not None:
    Copy_BOX_Current[Copy_BOX_Current[2]['R1']['location']].append(Copy_BOX_Current[2]['R1']['block'])
    Copy_BOX_Current[2]['R1']['block'] = None
    results.append(Copy_BOX_Current)

  # R2 Pickup
  Copy_BOX_Current = copy.deepcopy(state)
  if Copy_BOX_Current[2]['R2']['block'] is None and len(Copy_BOX_Current[Copy_BOX_Current[2]['R2']['location']]) > 0:
    Copy_BOX_Current[2]['R2']['block'] = Copy_BOX_Current[Copy_BOX_Current[2]['R2']['location']].pop()
    results.append(Copy_BOX_Current)

  # R2 DropOff
  Copy_BOX_Current = copy.deepcopy(state)
  if Copy_BOX_Current[2]['R2']['block'] is not None:
    Copy_BOX_Current[Copy_BOX_Current[2]['R2']['location']].append(Copy_BOX_Current[2]['R2']['block'])
    Copy_BOX_Current[2]['R2']['block'] = None
    results.append(Copy_BOX_Current)

  # Move
  Copy_BOX_Current = copy.deepcopy(state)
  if Copy_BOX_Current[2]['R1']['block'] is not None or Copy_BOX_Current[2]['R2']['block'] is not None:
    Copy_BOX_Current[2]['R1']['location'], Copy_BOX_Current[2]['R2']['location'] = Copy_BOX_Current[2]['R2']['location'], Copy_BOX_Current[2]['R1']['location']
    results.append(Copy_BOX_Current)

  return results

def find_index(value):
  for i, row in enumerate(BOX_Desired):
    if value in row:
      j = row.index(value)
      return i, j

def heuristic(current_state):
  total = 0
  for i in range(len(current_state)-1):
    for j in range(len(current_state[i])):
      a,b = find_index(current_state[i][j])
      if i == a:
        total += abs(j - b)
      else:
        if i == 0 and a == 1:
          total += abs(len(current_state[0]) - j) + abs(len(current_state[1]) - b)
        if i == 1 and a == 0:
          total += abs(len(current_state[1]) - j)  + abs(len(current_state[0]) - b)
  return total

def get_states(initial_state):
  fringe = [(heuristic(initial_state), [initial_state])]

  while fringe:
    _, current_path = heapq.heappop(fringe)
    current_state = current_path[-1]
    #print(heuristic(current_state))
    if current_state[0] == BOX_Desired[0] and current_state[1] == BOX_Desired[1]:
      return current_path

    if current_state in visited_states:
      continue

    visited_states.append(current_state)

    children = expand_children(current_state)

    for child in children:
      if child in visited_states:
        continue
      cost = len(current_path) + heuristic(child)
      updated_path = current_path + [child]
      heapq.heappush(fringe, (cost, updated_path))

states = get_states(BOX_Current)

def generateHtml():
    total_html = ''
    for i in range(len(states)):
        states[i][0].reverse()
        states[i][1].reverse()
        L1 = states[i][0]
        L2 = states[i][1]
        one = 2
        two = 1
        if states[i][2]['R1']['location'] == 0:
            one = 1
            two = 2
        html1 = f"""
        <div class="state-container">
            <h2>State {i}</h2>
            <div class="column-contatiner">
            <div class="location">
                <h2>R{one}</h2>
                <div class="robot-hand-container">
                    <div class="hand"></div>
                    <div class="palm"></div>
                    <div class="cube-container">
                    <div class="cube {'hide' if states[i][2][f'R{one}']['block'] == None else ''}"><span>{states[i][2][f'R{one}']['block']}</span></div>
                </div>
            </div>
        """
        total_html += html1

        for ab in L1:
            total_html += f"""
            <div class="cube-container">
              <div class="cube">
                <span>{ab}</span>
              </div>
            </div>
            """
        html2 = f"""
                <h2>L1</h2>
            </div>
        </div>
        <div class="column-contatiner">
        <div class="location">
            <h2>R{two}</h2>
            <div class="robot-hand-container">
                <div class="hand"></div>
                <div class="palm"></div>
                <div class="cube-container">
                    <div class="cube {'hide' if states[i][2][f'R{two}']['block'] == None else ''}"><span>{states[i][2][f'R{two}']['block']}</span></div>
                </div>
            </div>
        <div class="cube-container">
        """
        total_html += html2
        for ab in L2:
            total_html += f"""
            <div class="cube-container">
                <div class="cube">
                <span>{ab}</span>
            </div></div>"""
        html3 = """
            <h2>L2</h2>
          </div>
        </div>
    </div></div>"""
        total_html += html3

    styles = """
        .state-container{
        display: flex;
        flex-wrap: wrap;
        }

        .column-contatiner {
        display: flex;
        flex-direction: column;
        margin: auto;
        width: 100px;
        }


        .location {
        text-align: center;
        }

        .cube {
        width: 50px;
        height: 50px;
        background-color: #3498db;
        margin: auto;
        display: flow-root;
        margin-bottom: 5px;
        font-size: 35px;
        }

        .robot-hand-container {
        display: flex;
        flex-direction: column;
        margin-bottom: 20px;
        }

        .hand {
        width: 20px;
        height: 60px;
        background-color: #3498db;
        margin: auto;
        }

        .palm {
        width: 80px;
        height: 20px;
        margin: auto;
        background-color: #95a5a6;
        }

        .hide {
        visibility: hidden;
        }
    """
    base_html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>AI Output</title>
        <style>
            {styles}
        </style>
    </head>
        <body>
            { total_html }
        </body>
    </html>
"""
    return base_html

@app.route('/')
def index():

    full_html = render_template_string(generateHtml())
    return full_html

if __name__ == '__main__':
  app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat
